# 295. Find Median from Data Stream
**Difficulty:** 🔴 Hard · **Topic:** Heap · **LeetCode:** https://leetcode.com/problems/find-median-from-data-stream/

## 💡 Concepts

**Core concept(s):** Keep the numbers split into a **max-heap of the lower half** and a **min-heap of the upper half**.

**Why it applies here:** The median sits between the two halves. If the lower half's largest and the upper half's smallest are always at the top of two heaps (kept balanced in size), the median is one top (odd count) or the average of both tops (even count) — all in O(log n) per insert.

**Key intuition:** Balance a max-heap (low half) and a min-heap (high half); the median peeks at the tops.

---

### 📚 What is a Heap (Priority Queue)?
A **heap** always gives its smallest (min-heap) or largest (max-heap) item in **O(log n)**. Ideal for "keep the top K" or "always grab the current extreme".
- **In Python:** `heapq` (a min-heap; negate values for a max-heap).

---

**Prerequisite knowledge:**
- Two heaps kept balanced.
- Negating values for a max-heap in Python.

## 📝 Problem

Design a structure that supports `addNum(x)` and `findMedian()` on a growing stream.

**Example**
```
add 1, add 2 -> median 1.5 ; add 3 -> median 2
```

> One approach: two balanced heaps, `O(log n)` add, `O(1)` median.

### Approach — Two Heaps

**Idea:** `low` is a max-heap (store negatives) of the smaller half; `high` is a min-heap of the larger half. Push, then rebalance so their sizes differ by at most 1. The median is the top(s).

**Time:** `addNum` `O(log n)`, `findMedian` `O(1)`. **Space:** `O(n)`.

In [ ]:
import heapq

class MedianFinder:
    """Keep the numbers split into a low half (max-heap) and a high half (min-heap)."""
    def __init__(self):
        self.low = []     # max-heap of the smaller half (Python heapq is min, so store negatives)
        self.high = []    # min-heap of the larger half
    def addNum(self, num: int) -> None:
        heapq.heappush(self.low, -num)                        # tentatively add to the low half
        heapq.heappush(self.high, -heapq.heappop(self.low))   # move its largest into the high half
        if len(self.high) > len(self.low):                    # keep the halves balanced in size
            heapq.heappush(self.low, -heapq.heappop(self.high))
    def findMedian(self) -> float:
        if len(self.low) > len(self.high):
            return -self.low[0]                               # odd count -> top of the bigger half
        return (-self.low[0] + self.high[0]) / 2              # even count -> average of both tops

In [ ]:
# Correctness check
mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
assert mf.findMedian() == 1.5
mf.addNum(3)
assert mf.findMedian() == 2
import statistics, random
ref = []
mf2 = MedianFinder()
for _ in range(200):
    x = random.randint(0, 1000)
    ref.append(x); mf2.addNum(x)
    assert mf2.findMedian() == statistics.median(ref)
print("All tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

*(We time adding `n` numbers then reading the median — each add is O(log n).)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def run_stream(nums):
    mf = MedianFinder()
    for x in nums:
        mf.addNum(x)
    return mf.findMedian()

def make_worst_case(n):
    return ([(i * 7919) % 100000 for i in range(n)],)
solutions = {
    "two heaps: n adds O(n log n)": run_stream,
}
sizes = [20000, 40000, 80000, 160000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Two balanced heaps straddle the middle:** the median is always at the tops.
- **Max-heap via negation:** Python only has a min-heap, so store negatives for the lower half.
- **Signal:** "running median / middle of a stream", "k-th around the center over time".
- **Related problems:** Sliding Window Median, IPO, Kth Largest in a Stream.
- **Common pitfalls:** (1) letting the heaps get unbalanced; (2) wrong parity handling for even counts.